# Detecting & Classifying Cat Breeds

End-to-end transfer learning case study for cat breed recognition using a Colab runtime. We start from an images.cv export (`cats.zip`) that only guarantees a generic 'cat' label, so the notebook automatically infers breed labels from subfolders if they exist or creates pseudo-breed classes otherwise.

**Workflow overview:** unzip the dataset, explore the raw files, build stratified splits, train a transfer-learning model, and evaluate + demo predictions directly in Colab.


In [ ]:
# --- Step 1: Unzip cats.zip and inspect the folder tree ---
import os
import zipfile
from pathlib import Path

ZIP_PATH = Path('/content/cats.zip')
EXTRACT_DIR = Path('/content/cats')

if not ZIP_PATH.exists():
    raise FileNotFoundError('cats.zip was not found in /content. Upload it via the Colab sidebar first.')

if EXTRACT_DIR.exists() and any(EXTRACT_DIR.iterdir()):
    print(f"{EXTRACT_DIR} already exists; skipping extraction.")
else:
    EXTRACT_DIR.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
        zf.extractall(EXTRACT_DIR)
    print(f'Extracted {ZIP_PATH.name} -> {EXTRACT_DIR}')

os.chdir(EXTRACT_DIR)
print(f'Current working directory: {Path.cwd()}')

def print_tree(start_path: Path, max_depth: int = 2, max_files: int = 15):
    start_path = Path(start_path)
    for root, dirs, files in os.walk(start_path):
        rel = Path(root).relative_to(start_path)
        depth = len(rel.parts)
        if depth > max_depth:
            dirs[:] = []
            continue
        dirs[:] = sorted(dirs)
        label = '.' if rel == Path('.') else rel.as_posix()
        indent = '    ' * depth
        print(f'{indent}{label}/')
        file_list = sorted(files)
        for fname in file_list[:max_files]:
            print(f'{indent}    {fname}')
        if len(file_list) > max_files:
            remaining = len(file_list) - max_files
            print(f'{indent}    ... ({remaining} more files)')

print('\nDirectory snapshot (depth<=2):')
print_tree(Path.cwd(), max_depth=2)


In [ ]:
# --- Step 2: Install/verify dependencies (PyTorch, torchvision, sklearn, etc.) ---
%pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
%pip install -q matplotlib seaborn scikit-learn pandas tqdm

import shutil

if shutil.which('nvidia-smi'):
    !nvidia-smi
else:
    print('NVIDIA GPU not detected; continuing on CPU runtime.')


In [ ]:
# --- Step 3: Build dataset metadata + deterministic train/val/test splits ---
import random
from collections import Counter
from pathlib import Path
from typing import Sequence, Tuple

from PIL import Image
import torch
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from sklearn.model_selection import train_test_split

BASE_DIR = Path('/content/cats')
if not BASE_DIR.exists():
    raise FileNotFoundError('Expected /content/cats to exist after extraction.')

IMAGE_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.gif', '.webp'}
random.seed(42)

def has_image_files(folder: Path) -> bool:
    return any(file.suffix.lower() in IMAGE_EXTS for file in folder.rglob('*') if file.is_file())

def discover_label_dirs(root: Path):
    return [child for child in root.iterdir() if child.is_dir() and has_image_files(child)]

search_space = [BASE_DIR] + [p for p in BASE_DIR.rglob('*') if p.is_dir()]
dataset_root = BASE_DIR
label_dirs = []

for candidate in search_space:
    try:
        dirs = discover_label_dirs(candidate)
    except PermissionError:
        continue
    if len(dirs) > len(label_dirs):
        dataset_root = candidate
        label_dirs = dirs

if len(label_dirs) > 1:
    label_strategy = 'folder_structure'
    samples = []
    for class_dir in sorted(label_dirs):
        for img_path in class_dir.rglob('*'):
            if img_path.is_file() and img_path.suffix.lower() in IMAGE_EXTS:
                samples.append((img_path, class_dir.name))
else:
    label_strategy = 'pseudo_labels'
    flattened_pool = [p for p in BASE_DIR.rglob('*') if p.is_file() and p.suffix.lower() in IMAGE_EXTS]
    if not flattened_pool:
        raise RuntimeError('No image files were found under /content/cats. Check the zip contents.')
    random.Random(42).shuffle(flattened_pool)
    pseudo_classes = max(2, min(4, len(flattened_pool)))
    class_tokens = [f'cat_variant_{i}' for i in range(pseudo_classes)]
    samples = []
    for idx, img_path in enumerate(flattened_pool):
        label = class_tokens[idx % pseudo_classes]
        samples.append((img_path, label))
    label_dirs = []
    dataset_root = BASE_DIR

class_names = sorted({label for _, label in samples})
label_to_idx = {label: idx for idx, label in enumerate(class_names)}
IDX_TO_LABEL = {idx: label for label, idx in label_to_idx.items()}
indexed_samples = [(path, label_to_idx[label]) for path, label in samples]

def stratify_labels(items: Sequence[Tuple[Path, int]]):
    counts = Counter([lbl for _, lbl in items])
    return None if not counts or min(counts.values()) < 2 else [lbl for _, lbl in items]

train_split = 0.7
val_split = 0.15
test_split = 0.15

strat_labels = stratify_labels(indexed_samples)
train_samples, temp_samples = train_test_split(
    indexed_samples,
    test_size=(1 - train_split),
    random_state=42,
    stratify=strat_labels,
)

temp_strat = stratify_labels(temp_samples)
val_ratio_adjusted = val_split / (val_split + test_split)
val_samples, test_samples = train_test_split(
    temp_samples,
    test_size=(1 - val_ratio_adjusted),
    random_state=42,
    stratify=temp_strat,
)

class CatDataset(Dataset):
    def __init__(self, samples, transform=None):
        self.samples = samples
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        image = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        return image, label

IMAGE_SIZE = 224
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(IMAGE_SIZE, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.1, hue=0.02),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

eval_transform = transforms.Compose([
    transforms.Resize(IMAGE_SIZE + 32),
    transforms.CenterCrop(IMAGE_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

train_dataset = CatDataset(train_samples, transform=train_transform)
val_dataset = CatDataset(val_samples, transform=eval_transform)
test_dataset = CatDataset(test_samples, transform=eval_transform)

BATCH_SIZE = 32
NUM_WORKERS = 2

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

dataloaders = {'train': train_loader, 'val': val_loader, 'test': test_loader}
dataset_sizes = {
    'train': len(train_dataset),
    'val': len(val_dataset),
    'test': len(test_dataset),
}
CLASS_NAMES = class_names
NUM_CLASSES = len(CLASS_NAMES)
test_results = {'targets': [], 'preds': []}

print(f'Discovery strategy: {label_strategy}')
print(f'Dataset root: {dataset_root}')
print(f'Classes ({NUM_CLASSES}): {CLASS_NAMES}')
print('Split sizes:', dataset_sizes)


## Raw Data Insights

We now inspect class balance and visualize a quick image grid. Adjust the assumptions (e.g., folder names or pseudo-class count) if your zip uses a different hierarchy such as `animals/cat/<breed>`.


In [ ]:
# --- Step 4: Exploratory data analysis (class distribution + sample grid) ---
import itertools
import random
from collections import Counter

import matplotlib.pyplot as plt

all_split_samples = train_samples + val_samples + test_samples
label_counts = Counter([label for _, label in all_split_samples])
indices = list(range(NUM_CLASSES))
heights = [label_counts.get(idx, 0) for idx in indices]

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(indices, heights, color='#4f8cff')
ax.set_xticks(indices)
ax.set_xticklabels(CLASS_NAMES, rotation=45, ha='right')
ax.set_ylabel('Images')
ax.set_title('Image count per class')
plt.tight_layout()
plt.show()

grid_count = min(12, len(all_split_samples))
if grid_count:
    chosen = random.sample(all_split_samples, grid_count)
    rows = (grid_count + 3) // 4
    fig, axes = plt.subplots(rows, 4, figsize=(14, 3 * rows))
    axes = axes.flatten()
    for ax, sample in itertools.zip_longest(axes, chosen):
        if sample is None:
            ax.axis('off')
            continue
        img_path, label_idx = sample
        image = Image.open(img_path).convert('RGB')
        ax.imshow(image)
        ax.set_title(IDX_TO_LABEL[label_idx])
        ax.axis('off')
    plt.tight_layout()
else:
    print('No images available for visualization.')


In [ ]:
# --- Step 5: Define transfer-learning model + optimization components ---
import torch
import torch.nn as nn
from torchvision import models

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

weights = models.ResNet18_Weights.DEFAULT
model = models.resnet18(weights=weights)
model.fc = nn.Linear(model.fc.in_features, NUM_CLASSES)
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10)

print(model.fc)


In [ ]:
# --- Step 6: Training loop with train/val tracking ---
from copy import deepcopy
from tqdm import tqdm

EPOCHS = 5
history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
best_acc = 0.0
best_weights = deepcopy(model.state_dict())

for epoch in range(EPOCHS):
    print(f'Epoch {epoch + 1}/{EPOCHS}')
    for phase in ['train', 'val']:
        if phase == 'train':
            model.train()
        else:
            model.eval()
        running_loss = 0.0
        running_corrects = 0
        sample_count = 0
        loop = tqdm(dataloaders[phase], desc=f'{phase.title()} {epoch + 1}', leave=False)
        for inputs, labels in loop:
            inputs = inputs.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()
            with torch.set_grad_enabled(phase == 'train'):
                outputs = model(inputs)
                _, preds = torch.max(outputs, 1)
                loss = criterion(outputs, labels)
                if phase == 'train':
                    loss.backward()
                    optimizer.step()

            running_loss += loss.item() * inputs.size(0)
            running_corrects += torch.sum(preds == labels).item()
            sample_count += inputs.size(0)

        epoch_loss = running_loss / max(1, sample_count)
        epoch_acc = running_corrects / max(1, sample_count)
        history[f'{phase}_loss'].append(epoch_loss)
        history[f'{phase}_acc'].append(epoch_acc)
        print(f"{phase.title()} -> Loss: {epoch_loss:.4f} | Acc: {epoch_acc:.4f}")

        if phase == 'val':
            if scheduler:
                scheduler.step()
            if epoch_acc > best_acc:
                best_acc = epoch_acc
                best_weights = deepcopy(model.state_dict())
                torch.save(best_weights, 'best_model.pth')
                print(f'✨ Saved new best model (val acc={best_acc:.4f})')

model.load_state_dict(best_weights)
print('Loaded best validation checkpoint.')


In [ ]:
# --- Step 7: Plot training curves ---
import matplotlib.pyplot as plt

if len(history['train_loss']) == 0:
    print('History is empty; run the training cell first.')
else:
    epochs_range = range(1, len(history['train_loss']) + 1)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(epochs_range, history['train_loss'], label='Train')
    axes[0].plot(epochs_range, history['val_loss'], label='Val')
    axes[0].set_title('Loss')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss')
    axes[0].legend()
    axes[1].plot(epochs_range, history['train_acc'], label='Train')
    axes[1].plot(epochs_range, history['val_acc'], label='Val')
    axes[1].set_title('Accuracy')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Accuracy')
    axes[1].legend()
    plt.tight_layout()


In [ ]:
# --- Step 8: Evaluate on the held-out test set ---
from sklearn.metrics import accuracy_score, classification_report
from tqdm import tqdm

model.eval()
all_targets, all_preds = [], []
with torch.no_grad():
    for inputs, labels in tqdm(dataloaders['test'], desc='Testing', leave=False):
        inputs = inputs.to(device)
        labels = labels.to(device)
        outputs = model(inputs)
        probs = torch.softmax(outputs, dim=1)
        preds = torch.argmax(probs, dim=1)
        all_targets.extend(labels.cpu().tolist())
        all_preds.extend(preds.cpu().tolist())

if not all_targets:
    print('Test loader is empty; add more data and rerun the splits cell.')
else:
    test_acc = accuracy_score(all_targets, all_preds)
    print(f'Test accuracy: {test_acc:.4f}')
    print(classification_report(all_targets, all_preds, target_names=CLASS_NAMES, zero_division=0))

test_results = {'targets': all_targets, 'preds': all_preds}


In [ ]:
# --- Step 9: Confusion matrix visualization ---
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

if not test_results.get('preds'):
    print('Run the evaluation cell first to populate test_results.')
else:
    cm = confusion_matrix(test_results['targets'], test_results['preds'], labels=list(range(NUM_CLASSES)))
    fig, ax = plt.subplots(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('True')
    ax.set_title('Confusion Matrix')
    plt.tight_layout()


## Demo Predictions

Feed random cat photos from the dataset (or replace the paths with your own uploads) to inspect the model's top-1 prediction and confidence.


In [ ]:
# --- Step 10: Inference demo with random samples ---
import random
import matplotlib.pyplot as plt

model.eval()
demo_pool = test_samples if len(test_samples) >= 6 else train_samples
if not demo_pool:
    raise RuntimeError('No samples available for inference demo. Add more images and rerun the splits cell.')

k = min(6, len(demo_pool))
chosen = random.sample(demo_pool, k=k)
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.flatten()

for ax, (img_path, _) in zip(axes, chosen):
    image = Image.open(img_path).convert('RGB')
    tensor = eval_transform(image).unsqueeze(0).to(device)
    with torch.no_grad():
        probs = torch.softmax(model(tensor), dim=1).cpu().numpy()[0]
    pred_idx = int(probs.argmax())
    pred_label = IDX_TO_LABEL[pred_idx]
    confidence = probs[pred_idx] * 100
    ax.imshow(image)
    ax.set_title(f'{pred_label} ({confidence:.1f}%)')
    ax.axis('off')

for ax in axes[k:]:
    ax.axis('off')

plt.tight_layout()


## Conclusion & Limitations

This notebook delivers an end-to-end Colab workflow: data ingestion, exploratory analysis, transfer-learning with ResNet-18, evaluation, and inference demos for cat breed recognition. Because the original images.cv export only guarantees a single 'cat' category, any multi-class labels come from either (a) subfolders that already exist inside `cats.zip` or (b) pseudo-labels we deterministically created for instructional purposes. If you upload a richer dataset (e.g., true breed folders), simply re-run from the top and the pipeline will adapt automatically.
